In [1]:
# Cell 1: Clone repo + install deps
%cd /content
!git clone https://github.com/drosadocastro-bit/cibuco-boriken
%cd /content/cibuco-boriken
print("Setup complete")

/content
Cloning into 'cibuco-boriken'...
remote: Enumerating objects: 207, done.
remote: Counting objects: 100% (207/207), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 207 (delta 116), reused 134 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (207/207), 299.47 KiB | 1.66 MiB/s, done.
Resolving deltas: 100% (116/116), done.
/content/cibuco-boriken
Setup complete


In [2]:
!pip install -q tensorflow tensorflow-hub librosa



In [3]:
# Cell 2: Kaggle credentials (secure)
import os
import json
from google.colab import userdata

# Store token in Colab Secrets (never in code)
# Steps:
#   1. Click the 🔑 key icon in left sidebar
#   2. Add secret name: KAGGLE_USERNAME → drosadocastro-bit
#   3. Add secret name: KAGGLE_KEY → your_new_token
#   4. Enable notebook access for both

os.makedirs('/root/.kaggle', exist_ok=True)

kaggle_creds = {
    "username": userdata.get('kaggle_username'),
    "key": userdata.get('kaggle_key')
}

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle configured securely ✅")

# Download data
!pip install -q kaggle==1.6.17
!kaggle competitions download -c birdclef-2026
!mkdir -p data/birdclef-2026
!unzip -q birdclef-2026.zip -d data/birdclef-2026
print("Data ready ✅")


Kaggle configured securely ✅
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
100% 15.0G/15.0G [01:12<00:00, 201MB/s]
100% 15.0G/15.0G [01:12<00:00, 220MB/s]
Data ready ✅


In [4]:
# Cell 3: Mount Drive (save model after training)
import os
os.environ['BIRDCLEF_DATA_DIR'] = '/content/cibuco-boriken/data/birdclef-2026'

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import librosa
import numpy as np
from pathlib import Path
from tqdm import tqdm
import pandas as pd

DATA_DIR = Path('/content/cibuco-boriken/data/birdclef-2026')
SAMPLE_RATE = 32000

train_df = pd.read_csv(DATA_DIR / 'train.csv')
print(f'Training files: {len(train_df)}')

# Test single file first
test_file = DATA_DIR / 'train_audio' / train_df['filename'].iloc[0]
print(f'Test file: {test_file}')
print(f'Exists: {test_file.exists()}')

Training files: 35549
Test file: /content/cibuco-boriken/data/birdclef-2026/train_audio/1161364/iNat1216197.ogg
Exists: True


In [14]:
# Correct extraction
embedding = results[1].numpy()  # index 1 = embedding
logits = results[0].numpy()     # index 0 = species logits

print(f'Embedding shape: {embedding.shape}')  # (1, 1280)
print(f'Logits shape: {logits.shape}')        # (1, 10932)
print(f'Embedding sample: {embedding[0][:5]}')
print('Perch extraction working ✅')

Embedding shape: (1, 1280)
Logits shape: (1, 10932)
Embedding sample: [ 0.20427547 -0.02933899  0.02453168 -0.01707478 -0.02904083]
Perch extraction working ✅


In [15]:
from tqdm import tqdm
import os

WINDOW_SEC = 5.0
WINDOW_SAMPLES = int(WINDOW_SEC * SAMPLE_RATE)
SAVE_PATH = '/content/drive/MyDrive/cibuco_boriken/perch_train_embeddings.npz'

embeddings_dict = {}
errors = 0

for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc='Perch embeddings'):
    filepath = DATA_DIR / 'train_audio' / row['filename']

    try:
        audio, _ = librosa.load(str(filepath), sr=SAMPLE_RATE, mono=True)

        # Take best 5s window (highest energy)
        best_embedding = None
        best_energy = -1

        for start in range(0, len(audio) - WINDOW_SAMPLES + 1, WINDOW_SAMPLES):
            chunk = audio[start:start + WINDOW_SAMPLES]
            energy = np.mean(chunk ** 2)
            if energy > best_energy:
                best_energy = energy
                chunk_tf = tf.constant(chunk[np.newaxis, :], dtype=tf.float32)
                result = perch_model.infer_tf(chunk_tf)
                best_embedding = result[1].numpy()[0]  # (1280,)

        if best_embedding is not None:
            embeddings_dict[row['filename']] = best_embedding

    except Exception as e:
        errors += 1
        continue

# Save to Drive
np.savez_compressed(SAVE_PATH, **{
    k.replace('/', '_').replace('.', '_'): v
    for k, v in embeddings_dict.items()
})
print(f'Saved {len(embeddings_dict)} embeddings ✅')
print(f'Errors: {errors}')

Perch embeddings: 100%|██████████| 35549/35549 [19:52<00:00, 29.80it/s]


Saved 32948 embeddings ✅
Errors: 0


In [16]:
# Save filename mapping
import json

mapping = {
    k.replace('/', '_').replace('.', '_'): k
    for k in embeddings_dict.keys()
}
with open('/content/drive/MyDrive/cibuco_boriken/perch_embedding_mapping.json', 'w') as f:
    json.dump(mapping, f)
print(f'Mapping saved ✅')
print(f'Total embeddings: {len(embeddings_dict)}')
print(f'File size check incoming...')

import os
size = os.path.getsize('/content/drive/MyDrive/cibuco_boriken/perch_train_embeddings.npz')
print(f'Embeddings file: {size/1e6:.1f} MB')

Mapping saved ✅
Total embeddings: 32948
File size check incoming...
Embeddings file: 164.6 MB


In [ ]:
# Pull the fix
%cd /content/cibuco-boriken
!git pull origin main

# Re-run extraction
!python -m birdclef.perch_embed --extract \
    --model-dir perch_saved_model \
    --train-csv /content/cibuco-boriken/data/birdclef-2026/train.csv \
    --audio-dir /content/cibuco-boriken/data/birdclef-2026/train_audio \
    --output-dir perch_embeddings

/content/cibuco-boriken
From https://github.com/drosadocastro-bit/cibuco-boriken
 * branch            main       -> FETCH_HEAD
Already up to date.
2026-03-17 14:31:08.369591: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-17 14:31:08.377858: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773757868.387223   11287 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773757868.390275   11287 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registe

In [ ]:
!python -m birdclef.perch_classify \
    --embeddings-dir perch_embeddings \
    --epochs 50 \
    --patience 10 \
    --output perch_head.pt

birdclef.perch_classify | INFO | Loaded 118956 embeddings, 206 species
birdclef.perch_classify | INFO | Training PerchHead: 50 epochs, batch=256, lr=0.001
birdclef.perch_classify | INFO | Epoch   1/50 | Train Loss: 0.162739 | Val Loss: 0.032964 | Time: 1.3s
birdclef.perch_classify | INFO |   -> Saved best (val_loss=0.032964)
birdclef.perch_classify | INFO | Epoch   2/50 | Train Loss: 0.035327 | Val Loss: 0.032458 | Time: 1.1s
birdclef.perch_classify | INFO |   -> Saved best (val_loss=0.032458)
birdclef.perch_classify | INFO | Epoch   3/50 | Train Loss: 0.034898 | Val Loss: 0.032370 | Time: 0.9s
birdclef.perch_classify | INFO |   -> Saved best (val_loss=0.032370)
birdclef.perch_classify | INFO | Epoch   4/50 | Train Loss: 0.034709 | Val Loss: 0.033583 | Time: 0.9s
birdclef.perch_classify | INFO | Epoch   5/50 | Train Loss: 0.034472 | Val Loss: 0.031095 | Time: 0.9s
birdclef.perch_classify | INFO |   -> Saved best (val_loss=0.031095)
birdclef.perch_classify | INFO | Epoch   6/50 | Train 

In [ ]:
# SAVE MODEL IMMEDIATELY AFTER TRAINING
# Save to Drive
import shutil
shutil.copytree('perch_saved_model', '/content/drive/MyDrive/perch_saved_model', dirs_exist_ok=True)
shutil.copy('perch_head.pt', '/content/drive/MyDrive/perch_head.pt')
print('Saved to Drive ✅'),


Saved to Drive ✅


(None,)

In [ ]:
# Cell 5: CFAR Phase 2 k-sweep (3 conditions)
import os
os.environ['BIRDCLEF_DATA_DIR'] = '/content/cibuco-boriken/data/birdclef-2026'

!BIRDCLEF_DATA_DIR=/content/cibuco-boriken/data/birdclef-2026 \
  python -m birdclef.evaluate_thresholds \
  --backbone efficientnet_b2 \
  --include-soundscapes \
  --k-sweep 1.0 2.0 3.0 \
  --temperature 0.3

print("k-sweep complete ✅")

In [ ]:
# Cell 6: Display and save figure/model
from IPython.display import Image, display
import shutil

display(Image('k_sweep_figure.png'))

shutil.copy('k_sweep_figure.png', SAVE_DIR + 'k_sweep_500samples.png')
shutil.copy('birdclef/models/birdclef_model.pt', SAVE_DIR + 'birdclef_model_500samples.pt')
print("Saved to Drive")

In [ ]:
# Cell 7: Print paper-ready results table
import json
from pathlib import Path

results_path = Path('k_sweep_results.json')
if not results_path.exists():
    print('k_sweep_results.json not found. Run Cell 5 first.')
else:
    rows = json.loads(results_path.read_text(encoding='utf-8'))
    print('## Table 1: CFAR k-Sensitivity Results (500 samples)')
    print('| k | F1 Fixed | F1 CFAR | FPR Fixed | FPR CFAR | T_mean |')
    print('|---|----------|---------|-----------|----------|--------|')
    for r in rows:
        print(f"| {r['k']:.1f} | {r['f1_fixed']:.4f} | {r['f1_cfar']:.4f} | {r['fpr_fixed']:.4f} | {r['fpr_cfar']:.4f} | {r['threshold_mean']:.4f} |")

In [ ]:
# Temperature Goldilocks curve
import matplotlib.pyplot as plt
import numpy as np

temps  = [0.05,   0.10,   0.20,   0.30,   0.50]
deltas = [0.0001, 0.0003, 0.0012, 0.0019, -0.0394]
aucs   = [0.5934, 0.7525, 0.7582, 0.7582, 0.7582]
fprs   = [0.0003, 0.0003, 0.0004, 0.0006, 0.0011]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Figure 2: Temperature Scaling — Goldilocks Zone for CFAR')

# Left: F1 delta vs temperature
ax1.plot(temps, deltas, 'b-o', linewidth=2, markersize=8)
ax1.axhline(y=0, color='gray', linestyle='--', label='Fixed baseline')
ax1.axvline(x=0.3, color='green', linestyle='--', alpha=0.7, label='Optimal T=0.3')
ax1.fill_between(temps, deltas, 0,
                  where=[d>0 for d in deltas],
                  alpha=0.2, color='green', label='CFAR advantage')
ax1.fill_between(temps, deltas, 0,
                  where=[d<0 for d in deltas],
                  alpha=0.2, color='red', label='CFAR disadvantage')
ax1.set_xlabel('Temperature T')
ax1.set_ylabel('F1 Delta (CFAR - Fixed)')
ax1.set_title('F1 Improvement vs Temperature')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: AUC vs temperature
ax2.plot(temps, aucs, 'r-s', linewidth=2, markersize=8)
ax2.axvline(x=0.3, color='green', linestyle='--', alpha=0.7, label='Optimal T=0.3')
ax2.set_xlabel('Temperature T')
ax2.set_ylabel('ROC-AUC')
ax2.set_title('AUC vs Temperature')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('temperature_goldilocks.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 2 saved ✅")